In [29]:
# Import packages

## These are some essential Python packages for data science tasks. 
## They are widely used in various scripts because their combination allows to perform a powerful series of operations on data.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import os
import time
import math
import warnings
warnings.filterwarnings('ignore')

## There are packages related to working with geospatial data and performing spatial analysis. 
## They allow you to represent and manipulate geographical features and data

#import geopandas as gpd
import shapely
from shapely.geometry import LineString, Point, MultiLineString, Polygon, MultiPoint
from shapely.ops import linemerge, nearest_points, unary_union, split, substring, voronoi_diagram
from shapely import minimum_rotated_rectangle, unary_union
from scipy.spatial import Voronoi, cKDTree
import networkx as nx
from scipy.optimize import Bounds, differential_evolution

import tkinter as tk
from tkinter import filedialog, messagebox
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

In [ ]:
origin_crs = 'EPSG:4326'   ### WGS84 

In [ ]:
def process_buildings(target_crs):
    messagebox.showinfo('OnSSET', 'Open the GIS file with building polygons')
    buildings_path = filedialog.askopenfilename()
    buildings = gpd.read_file(buildings_path).to_crs(target_crs)
    buildings["area_in_meters"] = buildings.geometry.area
    
    return buildings

In [40]:
def calc_pv_gen(lat, lon, bulk=False):

    url = "https://re.jrc.ec.europa.eu/api/v5_3/seriescalc"

    params = {
        "lat": lat,
        "lon": lon,
        "startyear": 2022,
        "endyear": 2022,
        "outputformat": "json",
        "browser": 0,
        "angle": 0,       # 0 = horizontal surface
        "optimalangles": 1,
        "loss": 10,
        "global": 1,      # request GHI
        "temperature": 1,  # request air temperature
        "peakpower": 1,
        "pvtechchoice": "crystSi",
        "pvcalculation": 1
    }

    response = requests.get(url, params=params)
    data = response.json()
    
    # Extract hourly results
    hourly = data['outputs']['hourly']
    P_values = np.array([h["P"] / 1000 for h in hourly], dtype=float)
    
    # Generate the array for hours of the year\n"
    hours_of_year = np.arange(8760)  # Array from 0 to 8759\n",
    if bulk==False:
        # Create the plot
        plt.figure(figsize=(8, 4))
        plt.plot(hours_of_year, P_values, marker='', linestyle='-', color='b')
        # Labeling the axes
        plt.xlabel('Hours of the Year')
        plt.ylabel('Hourly PV output (kW / kWp)')
        # Adding a title
        plt.title('Hourly PV output (kW / kWp) for the community')
        # Show grid
        plt.grid(True)
        # Display the plot
        plt.show()
    
    return P_values

In [ ]:
def visualize_community(polygon):
    # Vizualizing the community in focus
    fig, ax = plt.subplots(figsize=(3, 3))
    cluster_polygons.plot(ax=ax, edgecolor='black', alpha=0.2)
    ax.set_aspect('equal', 'box')
    txt = ax.set_title('Community', size=8)
    ax.tick_params(axis='x', labelcolor='black', labelsize=6)
    ax.tick_params(axis='y', labelcolor='black', labelsize=6)

In [ ]:
def visualize_structures(buildings, cluster_polygons):
    # Vizualizing structures within the community in focus
    fig, ax = plt.subplots(figsize=(5, 5))
    cluster_polygons.plot(ax=ax, edgecolor='black', alpha=0.2)
    buildings.plot(ax=ax, edgecolor='red', alpha=0.2)
    ax.set_aspect('equal', 'box')
    txt = ax.set_title('Structures', size=8)
    ax.tick_params(axis='x', labelcolor='black', labelsize=6)
    ax.tick_params(axis='y', labelcolor='black', labelsize=6)

In [ ]:
#additional_values_storage = []

def optimizer_de(diesel_price,
                 hourly_ghi,
                 hourly_temp,
                 load_curve,
                 diesel_cost,  # diesel generator capital cost, USD/kW rated power
                 discount_rate,
                 n_chg,  # charge efficiency of battery
                 n_dis,  # discharge efficiency of battery
                 battery_cost,  # battery capital cost, USD/kWh of storage capacity
                 pv_cost,  # PV panel capital cost, USD/kW peak power
                 charge_controller,  # PV charge controller cost, USD/kW peak power, set to 0 if already included in pv_cost
                 pv_inverter,  # PV inverter cost, USD/kW peak power, set to 0 if already included in pv_cost
                 pv_life,  # PV panel expected lifetime, years
                 diesel_life,  # diesel generator expected lifetime, years
                 pv_om,  # annual OM cost of PV panels
                 diesel_om,  # annual OM cost of diesel generator
                 battery_inverter_cost,
                 battery_inverter_life,
                 dod_max,  # maximum depth of discharge of battery
                 battery_inverter_eff,  # inverter_efficiency
                 lpsp_max,  # maximum loss of load allowed over the year, in share of kWh
                 diesel_limit,
                 full_life_cycles,
                 start_year,
                 end_year,
                 pv_inverter_eff,
                 pv_inverter_life,
                 charge_controller_life,
                 c_rate_chg,
                 c_rate_dis,
                 battery_om,
                 battery_inverter_om,
                 ):

    demand = load_curve.sum()

    # The following lines defines the solution space for the Particle Swarm Optimization (PSO) algorithm
    battery_bounds = [0, 5 * demand / 365]
    pv_bounds = [0, 5 * max(load_curve)]
    if diesel_limit == 0:
        diesel_bounds = [0, 0]
    else:
        diesel_bounds = [0.5, max(load_curve)]
    
    min_bounds = np.array([pv_bounds[0], battery_bounds[0], diesel_bounds[0]])
    max_bounds = np.array([pv_bounds[1], battery_bounds[1], diesel_bounds[1]])
    bounds = Bounds(min_bounds, max_bounds)

    # Create a series of the hour numbers (0-24) for one year
    hour_numbers = np.empty(8760)
    for i in range(365):
        for j in range(24):
            hour_numbers[i * 24 + j] = i * 24 + j #j

    def opt_func(X):
        global additional_values_storage  # Use global variable to store additional values
        results = find_least_cost_option(X, hourly_temp, hourly_ghi, hour_numbers, load_curve, battery_inverter_eff, n_dis, 
                                         n_chg, dod_max, diesel_price, end_year, start_year, pv_cost, charge_controller, 
                                         pv_inverter, pv_inverter_eff, pv_om, diesel_cost, diesel_om, battery_inverter_life, 
                                         battery_inverter_cost, diesel_life, pv_life, battery_cost, discount_rate, lpsp_max, 
                                         diesel_limit, full_life_cycles, pv_inverter_life, charge_controller_life,
                                         c_rate_chg, c_rate_dis, battery_om, battery_inverter_om,)
        
        lcoe = results[0]
        additional_values_storage = results[1:]  # Capture additional values
                                       
        return lcoe

    minimizer_kwargs = {"method": "BFGS"}
    pv_init = sum(pv_bounds) / 2
    battery_init = sum(battery_bounds) / 2
    diesel_init = sum(diesel_bounds) / 2
    x0 = [pv_init, battery_init, diesel_init]
    
    result = differential_evolution(opt_func, bounds, popsize=15, init='latinhypercube')  # init='halton' on newer env

    best_solution = result.x
    lcoe = opt_func(best_solution)  # This call will also update additional_values_storage
    
    # 
    #    battery_life, pv, diesel, npc, fuel_usage, annual_om, pv_gen
    
    return {
        "best_solution": best_solution,
        "lcoe": lcoe,
        "unmet_demand_share": additional_values_storage[0],
        "ren_share": 1 - additional_values_storage[1],
        "investment": additional_values_storage[2],
        "fuel_cost": additional_values_storage[3],
        "om_cost": additional_values_storage[4],
        "battery_life": additional_values_storage[6],
        "npc": additional_values_storage[9],
        "fuel_usage": additional_values_storage[10],
        "annual_om": additional_values_storage[11],
        "excess_gen": additional_values_storage[13],
        "pv_gen": additional_values_storage[12],
        "battery_soc": additional_values_storage[14],
        "net_load": additional_values_storage[15]
    }

In [ ]:
def distribution_network(buildings, cluster_polygons, target_crs, cost_grid_km, con_cost, cost_pole, grid_om, service_drop_line):
    #The households layer was previously uploaded as Polygon Layer. 
    #The following lines changed the geometry to point and defines a new layer *building_centroids* with the *target_crs* for the study area
    
    building_centroids_crs = buildings.copy()
    building_centroids_crs["geometry"] =  buildings['geometry'].centroid
    building_centroids = building_centroids_crs.to_crs(target_crs)
    
    ### This process outlines a method for generating a primary trunk line for settlements represented by polygon boundaries. 
    #1. **Boundary Points**: The user defines a spacing distance (default 50 meters) to create a set of evenly spaced points along the settlement polygon boundary.
    #2. **Voronoi Polygons**: These points are used to divide the settlement polygon into smaller regions called Voronoi polygons.
    #3. **Initial Trunk Line**: Edges of the Voronoi polygons are extracted, but any edge touching the settlement boundary is removed. This creates a preliminary version of the primary trunk line.
    #4. **Simplifying the Trunk Line**: Intersections where the trunk line branches off are identified. Only segments of the trunk line meeting these criteria are kept:
    #    * Located between two identified intersections.
    #    * Longer than a user-specified distance threshold.
    #5. **Splitting the Trunk Line**: The identified intersections are used to split the primary trunk line into separate segments. Any remaining long segments exceeding a user-defined distance threshold are also split.
    ###
    
    polygon_data = cluster_polygons.to_crs(target_crs)
    polygon_data = polygon_data.iloc[0]
    polygon = polygon_data['geometry']
    
    # Create trunk line(s)
    trunk_lines = create_trunk_line(polygon, spacing=150, plot=False)

    # Simplify the trunk line(s)
    trunk_lines, intersects = simplify_trunk_lines(trunk_lines, length_removal=200, split_distance=750, plot=False)

    trunk_lines_gdf = gpd.GeoDataFrame(geometry=trunk_lines)
    
    # Identify which part of the polygon belongs to which part of the trunk line(s)
    voronois = voronoi_areas(trunk_lines_gdf, polygon, plot=False)
    
    ## Pole locations
    
    #With the voronoi areas, now the focus is on identifying poles to where households will be connected. In each sub-area of the settlement polygon:
    #1. A mesh of evenly spaced points (default distance = 50 m) are created
    #2. Points are aligned to the x- and y-axis of the minimum bounding rectangle that can encapsulate the sub-area.
    #3. Any point that is close to the primary trunk line (default distance = 25 m) is excluded, as it is assumed that buildings within this distance would connect to a pole along the primary trunk line.
    #4. Buildings are assigned to the nearest of the candidate poles created in the previous step.
    #5. Finally, a Minimum Spanning Tree is created using NetworkX, to identify the secondary lines. The weights of the MST are assigned to favor in order:
    #    +  lines that are (near) orthogonal to the primary trunk
    #    + lines that are (near) parallel to the primary trunk 
    #    + diagonal lines 
    
    # The MST is run iteratively, testing the removal of poles to which no building was assigned until the shortest MST has been achieved (i.e. no more of the poles without households can be removed).
    
    trunk_p = []
    assigned_p = []
    lv_l = []
    service_l = []
    mst_p = []
    all_p = []
    long_services = 0

    multi_trunks = []
    multi_trunks_len = []

    multi_secondary = []
    multi_secondary_len = []

    multi_service = []
    multi_service_len = []

    multi_poles = []
    multi_all_poles = []

    # Iterate over each part of the trunk line 
    for id in range(len(trunk_lines)):
        all_poles, trunk_poles, poles, angle_radians_w, angle_radians_l = create_candidate_poles(voronois[id], trunk_lines[id], 
                                                                                                 pole_dist, buffer=15, plot=False)

        # print(trunk_poles)

        trunk_p += trunk_poles
        all_p += all_poles

        polygon_building = building_centroids.clip(voronois[id])

        # Ensure households are not MultiPoint
        polygon_building['geometry'] = polygon_building['geometry'].apply(convert_multipoint_to_point)

        assigned_poles, service_drops = assign_households(all_poles, polygon_building)

        assigned_p += all_poles
        service_l += service_drops

        for s in service_drops:
            if s.length > 70:
                long_services += 1

        weight = 0.5  # Weighting factor for the MST

        lv_lines, mst_poles = lv_lines_mst(all_poles, trunk_poles, assigned_poles, angle_radians_w, angle_radians_l, weight, plot=False)

        lv_gdf=gpd.GeoSeries(lv_lines)
        lv_gdf = lv_gdf.buffer(10)
        all_pole_gdf=gpd.GeoSeries(all_poles)
        intersecting_points = []
        # Loop over each point in points_gs
        for point in all_pole_gdf:
            # Check if this point intersects with any line in lines_gs
            if lv_gdf.intersects(point).any():
                intersecting_points.append(point)
        for p in trunk_poles:
            if p not in intersecting_points:
                intersecting_points.append(p)

        lv_l += lv_lines
        mst_p += intersecting_points # mst_poles
        
        # Geometry results from the sizing model (distribution networ and poles) are appended into a list and then store in geopandas format.
        
        multi_all_poles.append(MultiPoint(all_p))

    multi_trunks.append(MultiLineString(trunk_lines))
    multi_trunks_len.append(MultiLineString(trunk_lines).length)

    multi_secondary.append(MultiLineString(lv_l))
    multi_secondary_len.append(MultiLineString(lv_l).length)

    multi_service.append(MultiLineString(service_l))
    multi_service_len.append(MultiLineString(service_l).length)

    multi_poles.append(MultiPoint(mst_p)) 
    
    trunks_gdf = gpd.GeoDataFrame()
    trunks_gdf['Length'] = multi_trunks_len
    trunks_gdf['Type'] = "Trunk Line"
    trunks_gdf.geometry = multi_trunks
    trunks_gdf.set_crs(target_crs, inplace=True)
    #trunks_gdf['Community'] = community

    secondary_gdf = gpd.GeoDataFrame()
    secondary_gdf['Length'] = multi_secondary_len
    secondary_gdf['Type'] = "Secondary Line"
    secondary_gdf.geometry = multi_secondary
    secondary_gdf.set_crs(target_crs, inplace=True)
    #secondary_gdf['Community'] = community

    service_gdf = gpd.GeoDataFrame()
    service_gdf['Length'] = multi_service_len
    service_gdf['Type'] = "Service Line"
    service_gdf.geometry = multi_service
    service_gdf.set_crs(target_crs, inplace=True)
    #service_gdf['Community'] = community

    poles_gdf = gpd.GeoDataFrame()
    poles_gdf.geometry = multi_poles
    for i, multi_pole in enumerate(multi_poles):
        num_poles = len(multi_pole.geoms)
    poles_gdf['No. Poles'] = num_poles
    poles_gdf.set_crs(target_crs, inplace=True)
    #poles_gdf['Community'] = community
    poles_gdf.to_crs(4326, inplace=True)
    
    # Finally, all the individual MultiLineString GeoDataGrames can be merged into a single GeoDataFrame for further analysis.
    
    total_grid_gdf = gpd.GeoDataFrame(pd.concat([trunks_gdf, secondary_gdf, service_gdf], ignore_index=True)) # ToDo add poles_gdf???
    
    cost_grid = cost_grid_km * (sum(trunks_gdf["Length"])/1000 + sum(secondary_gdf["Length"])/1000)
    cost_service_drops = service_drop_line*(sum(service_gdf['Length'])/1000)
    connection_cost = con_cost * len(buildings)
    total_cost_poles = cost_pole * sum(poles_gdf["No. Poles"])
    total_cost_grid = cost_grid + cost_service_drops + total_cost_poles + connection_cost
    total_om_grid = grid_om * total_cost_grid
    
    return total_grid_gdf, trunks_gdf, secondary_gdf, service_gdf, poles_gdf, total_cost_grid, total_om_grid

In [1]:
def summary_results(ret, poles_gdf, trunks_gdf, secondary_gdf, service_gdf, lcoe_distribution, investment_other,
                   total_cost_labour, transportation_costs, other_om, discounted_costs, cluster_polygons):
    results = {
        "Name": ["PV Capacity [kWp]",
                 "Battery capacity [kWh]", 
                 "Diesel Capacity [kW]",

                 "Trunk Line [km]",
                 "Laterals [km]",
                 "Service lines [km]",
                 "No Poles",

                 "CAPEX generation [USD]",
                 "CAPEX Distribution [USD]",
                 "CAPEX Other [USD]",
                 "Cost of Labour [USD]",
                 "Cost of Transport [USD]",
                 "Total CAPEX [USD]",

                 "OPEX Generation [USD]",
                 "OPEX Distribution [USD]",
                 "OPEX Other [USD]",
                 "Total OPEX [USD]",
                 #"Annual OPEX [USD]",
                 "NPC Generation [USD]",
                 "NPC Distribution [USD]",
                 "NPC Other [USD]",
                 "NPC Total [USD]",

                 "LCOE Generation [USD/kWh]",
                 "LCOE Distribution [USD/kWh]",
                 "LCOE Other [USD/kWh]",
                 "LCOE Total [USD/kWh]",

                ],


        "Value": [round(ret["best_solution"][0],3), #PV_Capacity
                  round(ret["best_solution"][1],3),  #Battery Capacity
                  round(ret["best_solution"][2],3), #Diesel Capacity

                  round(sum(trunks_gdf["Length"])/1000,3), #kilometers_grid_main_line
                  round(sum(secondary_gdf["Length"])/1000,3), #kilometers_grid_secondary_lines
                  round(sum(service_gdf["Length"])/1000,3),#kilometers_services_drops
                  sum(poles_gdf["No. Poles"]), #Number of poles

                  round(ret["investment"], 0), # CAPEX of generation
                  round(lcoe_distribution[1],0), # CAPEX of distribution
                  round(investment_other, 0), # CAPEX of other (transport, labor, power-house) 
                  total_cost_labour, #Cost of Labour
                  transportation_costs, #Cost of transport
                  round(ret["investment"], 0) + round(lcoe_distribution[1],0) + round(investment_other, 0), #Total_capex

                  round(ret['fuel_cost'],0) + round(ret['om_cost'],0), #Total OPEX Generation
                  round(lcoe_distribution[2],0), #Total OPEX Distribution
                  round(other_om),
                  round(ret['fuel_cost'],0) + round(ret['om_cost'],0) + round(lcoe_distribution[2],0) + round(other_om),
                  round(ret["npc"], 0), # NPC of generation
                  round(lcoe_distribution[2],0), # NPC of distribution
                  round(discounted_costs, 0), # NPC of other
                  round(ret["npc"] + lcoe_distribution[2] + discounted_costs, 0),

                  round(ret["lcoe"],3),
                  round(lcoe_distribution[0], 3),
                  round(lcoe_other, 3),
                  round(ret["lcoe"],3) + round(lcoe_distribution[0], 3) + round(lcoe_other, 3),

                 ]
    }

    results_pd = pd.DataFrame(results, columns=["Name", "Value"])

    # Assign result columns to the original community aplha shape
    for key, value in results_pd.set_index('Name')['Value'].to_dict().items():
        cluster_polygons[key] = value

    return results_pd

In [2]:
def calculate_demand(smeshare, bundleCshare, bundleBshare, cluster_polygons, buildings, i=0, bulk=False):
    
    # Define bins and labels
    bins = [0, 20, 50, 150, float('inf')]
    labels = ['<20 m2', '20-50 m2', '50-150 m2', '>150 m2']

    # Calculate the statistics without adding a new column
    area_brackets = pd.cut(buildings['area_in_meters'], bins=bins, labels=labels, right=False)
    bracket_stats = area_brackets.value_counts().sort_index()

    # Add the total sum as value per column
    for label in labels:
        cluster_polygons[label] = bracket_stats[label]
        
    if bulk==False:
        print ("Total structures:", buildings.shape[0])
        print (bracket_stats)

    reasonablecons = 1        ## Connectivity rate of structures - for now this value should not be changed 

    cluster_polygons["Connections_All"] = np.ceil(cluster_polygons["<20 m2"] + 
                                   cluster_polygons["20-50 m2"]+ 
                                   cluster_polygons["50-150 m2"]+ 
                                   cluster_polygons[">150 m2"]).astype(int)

    cluster_polygons["Potential_Con"] = np.ceil(cluster_polygons["Connections_All"]*reasonablecons)

    def assignPUoE(a, b, c, d, e):
        puoe = int(round(-2.913362564 + (-0.00415764*a) + (0.008440603*b) + (0.016433547*c) + (0.722331464*d) + (-0.040674586*e),0))
        if puoe <=0:
            return 0
        else:
            return puoe

    cluster_polygons["PUoE"] = cluster_polygons.apply(lambda row: assignPUoE(row["<20 m2"],
                                                 row["20-50 m2"],
                                                 row["50-150 m2"],
                                                 row[">150 m2"],
                                                 0), axis=1)

    cluster_polygons["SME"] = np.ceil((cluster_polygons["Potential_Con"] - cluster_polygons["PUoE"])*smeshare)

    cluster_polygons["ResC"] = np.ceil((cluster_polygons["Potential_Con"] - cluster_polygons["PUoE"])*bundleCshare)

    cluster_polygons["ResB"] = np.ceil((cluster_polygons["Potential_Con"] - cluster_polygons["PUoE"])*bundleBshare)

    cluster_polygons["ResA"] = (cluster_polygons["Potential_Con"] - cluster_polygons["PUoE"] - cluster_polygons["SME"] - cluster_polygons["ResC"]- cluster_polygons["ResB"])

    #load_curve = calc_load_curve(tier_load, total_demand)
    load_curve = calc_load_curve(cluster_polygons["ResA"][i],
                                 cluster_polygons["ResB"][i],
                                 cluster_polygons["ResC"][i],
                                 cluster_polygons["SME"][i],
                                 cluster_polygons["PUoE"][i])

    return load_curve